In [ ]:

import os
from datetime import timedelta, timezone
from pathlib import Path
from fastapi import (
    APIRouter,
    UploadFile,
    File,
    HTTPException,
    status,
    Depends,
    BackgroundTasks,
)

from app.api.auth import get_current_user
from app.core.config import settings, PROJECT_ROOT
from app.utils.helpers import (
    generate_id,
    utc_now,
    record_activity,
    ensure_directory,
    safe_filename,
)


router = APIRouter(
    prefix="/materials",
    tags=["Materials"],
)


# Browsers/OSes report PDFs inconsistently; the filename extension and
# the %PDF header are checked as well.
ALLOWED_CONTENT_TYPES = {
    "application/pdf",
    "application/x-pdf",
    "application/octet-stream",
    "binary/octet-stream",
}

MAX_FILE_SIZE = 50 * 1024 * 1024


def get_database():
    """Return the application's MongoDB wrapper."""

    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    """Verify that a project belongs to the authenticated user."""

    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def material_response(material: dict) -> dict:
    """Remove MongoDB's internal ObjectId from API responses."""

    response = dict(material)

    response.pop("_id", None)

    return response


# A job that has been "processing" longer than this is assumed dead
# (server restarted / killed mid-job) and is surfaced as failed so the
# UI stops spinning and offers a retry.
STALE_PROCESSING_MINUTES = float(
    os.environ.get("MATERIAL_PROCESSING_TIMEOUT_MINUTES", "10")
)

INTERRUPTED_MESSAGE = (
    "Processing was interrupted (the server restarted or ran out of "
    "memory). Click Retry, or upload the file again."
)

TIMEOUT_MESSAGE = (
    "Processing did not finish in time. Click Retry, or upload the "
    "file again."
)


def _as_utc(value):
    """MongoDB returns naive datetimes; treat them as UTC."""

    if value is None:
        return None

    if value.tzinfo is None:
        return value.replace(tzinfo=timezone.utc)

    return value


def fail_stale_materials(database, **scope) -> int:
    """
    Mark overdue "processing" materials as failed.

    Without this a job whose worker died stays "processing" forever and
    the UI polls it indefinitely.
    """

    cutoff = utc_now() - timedelta(minutes=STALE_PROCESSING_MINUTES)

    materials = database.collection("materials")

    query = {"status": "processing", **scope}

    fixed = 0

    for document in materials.find(
        query,
        {
            "_id": 0,
            "id": 1,
            "created_at": 1,
            "processing_started_at": 1,
        },
    ):

        started = _as_utc(
            document.get("processing_started_at")
            or document.get("created_at")
        )

        if started is None or started > cutoff:
            continue

        materials.update_one(
            {"id": document["id"], "status": "processing"},
            {
                "$set": {
                    "status": "failed",
                    "processing_status": "failed",
                    "processing_error": TIMEOUT_MESSAGE,
                    "updated_at": utc_now(),
                }
            },
        )

        fixed += 1

    return fixed


def recover_interrupted_materials(database) -> int:
    """Run once at startup: nothing can still be processing after a restart."""

    result = database.collection("materials").update_many(
        {"status": "processing"},
        {
            "$set": {
                "status": "failed",
                "processing_status": "failed",
                "processing_error": INTERRUPTED_MESSAGE,
                "updated_at": utc_now(),
            }
        },
    )

    return result.modified_count


def process_material_background(
    material_id: str,
):
    """
    Process an uploaded material in the background.

    Defined as a normal function so Starlette runs it in a worker
    thread: PDF parsing and embedding are blocking CPU work and
    must not run on the event loop.
    """

    database = get_database()
    material = None

    try:
        # ----------------------------------------------------
        # Load material
        # ----------------------------------------------------
        material = database.collection(
            "materials"
        ).find_one(
            {
                "id": material_id,
            },
            {
                "_id": 0,
            },
        )

        if material is None:
            raise ValueError(
                f"Material not found: {material_id}"
            )

        # ----------------------------------------------------
        # Validate stored file path
        # ----------------------------------------------------
        file_path = material.get("file_path")

        if not file_path:
            raise ValueError(
                f"Material {material_id} has no file_path."
            )

        # ----------------------------------------------------
        # Delegate to DocumentWorker
        # ----------------------------------------------------
        from app.workers.document_worker import (
            DocumentJob,
            DocumentWorker,
        )

        worker = DocumentWorker(
            database=database
        )

        job = DocumentJob(
            material_id=material_id,
            file_path=str(file_path),
        )

        result = worker.process(job)

        record_activity(
            database,
            user_id=material["user_id"],
            project_id=material["project_id"],
            event_type="MATERIAL_PROCESSED",
            description=f"Processed material: {material.get('file_name', material_id)}",
            entity_type="material",
            entity_id=material_id,
        )

        return result

    except Exception as exc:
        # ----------------------------------------------------
        # Record processing failure
        # ----------------------------------------------------
        database.collection(
            "materials"
        ).update_one(
            {
                "id": material_id,
            },
            {
                "$set": {
                    "status": "failed",
                    "processing_status": "failed",
                    "processing_error": str(exc),
                    "updated_at": utc_now(),
                }
            },
        )

        if material is not None:
            record_activity(
                database,
                user_id=material.get("user_id"),
                project_id=material.get("project_id"),
                event_type="MATERIAL_PROCESSING_FAILED",
                description=f"Failed to process material: {material_id}",
                entity_type="material",
                entity_id=material_id,
                metadata={"error": str(exc)},
            )

        raise


@router.post(
    "/upload",
    status_code=status.HTTP_201_CREATED,
)
async def upload_material(
    background_tasks: BackgroundTasks,
    project_id: str,
    file: UploadFile = File(...),
    current_user=Depends(get_current_user),
):
    """
    Upload a PDF into an authorized project.
    The file is persisted immediately and document processing is
    scheduled in the background.
    """

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    if file.content_type not in ALLOWED_CONTENT_TYPES:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail="Only PDF files are supported.",
        )

    original_name = file.filename or "document.pdf"

    if not original_name.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail="Only PDF files are supported.",
        )

    filename = safe_filename(original_name)

    if not filename:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Invalid filename.",
        )

    upload_dir = Path(settings.upload_dir)

    if not upload_dir.is_absolute():
        upload_dir = PROJECT_ROOT / upload_dir

    ensure_directory(upload_dir)

    material_id = generate_id()

    stored_filename = f"{material_id}_{filename}"

    file_path = upload_dir / stored_filename

    content = await file.read()

    if len(content) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=status.HTTP_413_REQUEST_ENTITY_TOO_LARGE,
            detail="File exceeds the 50 MB limit.",
        )

    if not content:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Uploaded file is empty.",
        )

    if b"%PDF-" not in content[:1024]:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail="The uploaded file is not a valid PDF.",
        )

    file_path.write_bytes(content)

    now = utc_now()

    material = {
        "id": material_id,
        "user_id": current_user.id,
        "project_id": project_id,
        "file_name": original_name,
        "stored_file_name": stored_filename,
        "file_path": str(file_path),
        "file_type": "application/pdf",
        "file_size": len(content),
        "status": "processing",
        "processing_status": "processing",
        "processing_error": None,
        "processing_started_at": now,
        "created_at": now,
        "updated_at": now,
    }

    try:
        database.collection("materials").insert_one(material)

        record_activity(
            database,
            user_id=current_user.id,
            project_id=project_id,
            event_type="MATERIAL_UPLOADED",
            description=f"Uploaded material: {original_name}",
            entity_type="material",
            entity_id=material_id,
            metadata={
                "file_name": original_name,
                "file_size": len(content),
            },
        )
    except Exception:
        if file_path.exists():
            file_path.unlink()

        raise

    background_tasks.add_task(
        process_material_background,
        material_id,
    )

    return material_response(material)


@router.get(
    "/project/{project_id}",
)
def list_materials(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """List materials belonging to an authorized project."""

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    fail_stale_materials(
        database,
        project_id=project_id,
        user_id=current_user.id,
    )

    documents = database.collection("materials").find(
        {
            "project_id": project_id,
            "user_id": current_user.id,
        }
    ).sort(
        "created_at",
        -1,
    )

    return [
        material_response(document)
        for document in documents
    ]


@router.get(
    "/{material_id}",
)
def get_material(
    material_id: str,
    current_user=Depends(get_current_user),
):
    """Return metadata and processing status for an authorized material."""

    database = get_database()

    fail_stale_materials(
        database,
        id=material_id,
        user_id=current_user.id,
    )

    material = database.collection("materials").find_one(
        {
            "id": material_id,
            "user_id": current_user.id,
        }
    )

    if material is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Material not found.",
        )

    return material_response(material)


@router.post(
    "/{material_id}/retry",
)
def retry_material(
    material_id: str,
    background_tasks: BackgroundTasks,
    current_user=Depends(get_current_user),
):
    """Re-run processing for a failed or stuck material."""

    database = get_database()

    fail_stale_materials(
        database,
        id=material_id,
        user_id=current_user.id,
    )

    materials = database.collection("materials")

    material = materials.find_one(
        {
            "id": material_id,
            "user_id": current_user.id,
        }
    )

    if material is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Material not found.",
        )

    if material.get("status") == "processing":
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="This material is still being processed.",
        )

    file_path = material.get("file_path")

    if not file_path or not Path(file_path).exists():
        raise HTTPException(
            status_code=status.HTTP_410_GONE,
            detail=(
                "The original file is no longer on the server. "
                "Delete this entry and upload the PDF again."
            ),
        )

    now = utc_now()

    materials.update_one(
        {"id": material_id, "user_id": current_user.id},
        {
            "$set": {
                "status": "processing",
                "processing_status": "processing",
                "processing_error": None,
                "processing_started_at": now,
                "updated_at": now,
            }
        },
    )

    background_tasks.add_task(
        process_material_background,
        material_id,
    )

    return material_response(
        materials.find_one({"id": material_id})
    )


@router.delete(
    "/{material_id}",
    status_code=status.HTTP_204_NO_CONTENT,
)
def delete_material(
    material_id: str,
    current_user=Depends(get_current_user),
):
    """Delete an authorized material and its local file."""

    database = get_database()

    materials = database.collection("materials")

    material = materials.find_one(
        {
            "id": material_id,
            "user_id": current_user.id,
        }
    )

    if material is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Material not found.",
        )

    file_path = material.get("file_path")

    if file_path:
        try:
            path = Path(file_path)

            if path.exists():
                path.unlink()

        except Exception:
            pass

    database.collection("document_chunks").delete_many(
        {
            "material_id": material_id,
            "user_id": current_user.id,
        }
    )

    result = materials.delete_one(
        {
            "id": material_id,
            "user_id": current_user.id,
        }
    )

    if result.deleted_count == 0:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Material not found.",
        )

    return None
